Cel: Zrozumieć, które czynniki (kliniczne i styl życia) mają największy związek z cukrzycą oraz przygotować dane pod model predykcyjny.

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

In [ ]:
# Ustawienia widoku tabel
#max. 200 kolumn
pd.set_option("display.max_columns", 200)
#szerokosc wyswietlania tabel na 160 znakow
#by tabelki byly widoczne nieurwane
pd.set_option("display.width", 160)


In [ ]:
# Styl wykresów - biale tlo i siatka
sns.set_theme(style="whitegrid")
#stala liczba by wyniki losowe byly powtarzalne, taki sam podzial train/test po ponownym odpaleniu kodu
RANDOM_STATE = 42

In [ ]:
##wczytanie danych
DATA_FILE = "diabetes_prediction_dataset.csv"
#jesli sie uda - w colab jak nie - inne srodowisko
try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB and not Path(DATA_FILE).exists():
    print("Wgraj plik diabetes_prediction_dataset.csv")
    files.upload()
#wczytanie csv do dataframe
df = pd.read_csv(DATA_FILE)
#df.shape - ile wierszy i kolumn, df.head - pierwsze wiersze
print("Wczytano dane.")
print("Kształt (wiersze, kolumny):", df.shape)
display(df.head())

Szybkie poznanie danych:


*   Sprawdzenie typów danych
*   Sprawdzenie statystyki
*   Sprawdzenie unikalnych wartości




In [ ]:
print("\nInformacje o typach, brakach i pamięci")
df.info()
print("\nDescribe - numeryczne")
display(df.describe().T)
print("\nDescribe - kategoryczne")
display(df.describe(include="all").T)
#nunique - ile unikalnych wartosci ma kazda kolumna
#sort_values - od największej do najmniejszej liczby unikalnych
print("\nLiczba wartosci unikalnych ")
display(df.nunique().sort_values(ascending=False))

Ocena jakosci danych:


*   Braki danych - ile i gdzie

*   Wzorce braków - czy braki są losowe

*   Duplikaty - czy rekordy sa powielone

*   Walidacja wykresow klinicznych - czy wartosci sa realistyczne

*   Outliery - czy wystepują i ile ich jest jak występują



In [ ]:
###Braki danych - % i liczba
#df.isna() - robi tabele true/false gdzie true = brak , na_count- ile brakow w kazdej kolumnie
na_count = df.isna().sum().sort_values(ascending=False)
#.mean() - srednia true/false w kolumnie
#true=1.false=0. to srednia = % brakow
#sortowanie malejąco
#na_percentage - %brakow w kazdej kolumnie
na_percentage = (df.isna().mean() * 100).sort_values(ascending=False)

In [ ]:
#laczenie liczby i % w 1 tabele
missing_summary = pd.DataFrame({"na_count": na_count, "na_percentage": na_percentage})
print(" Braki danych (tylko kolumny z brakami)")
#pokazuje tylko te kolumny gdzie braki sa > 0
display(missing_summary[missing_summary["na_count"] > 0])


In [ ]:
# Mapa braków - wizualnie
plt.figure(figsize=(12, 3))
sns.heatmap(df.isna(), cbar=False)
plt.title("Mapa braków danych (True = brak)")
plt.show()

In [ ]:
if "diabetes" in df.columns:
    for col in df.columns:
        if df[col].isna().any():
            print(f"\n Wzorzec braków: {col} vs diabetes")
            display(
                #tworzenie tymaczasowej kolumny _na true/false - czy w tej kolumnie jest brak
                df.assign(_na=df[col].isna())
                #grupowanie po tymaczasowej kolumnie_na i diabetes
                  .groupby(["_na", "diabetes"])
                  #ile rekordow jest w kazdej kolumnie
                  .size()
                  #kolumny zamiast indeksow - czytelna tabelka
                  .unstack(fill_value=0)
            )

Duplikaty

In [ ]:
#true dla wierszy ktore sa identyczne jak wczesniejsze i ile takij jest sum()
dup_count = df.duplicated().sum()
print("Duplikaty (dokładnie identyczne wiersze)")
print("Liczba duplikatów:", dup_count)

if dup_count > 0:
    display(df[df.duplicated()].head(10))

In [ ]:
#usuwanie duplikatow
#resetowanie indexu
df = df.drop_duplicates().reset_index(drop=True)
print(" Po usunięciu duplikatów:", df.shape)

Walidacja zakresów klinicznych + clipping

In [ ]:
# Funkcja tworząca flagi wartości "poza sensownym zakresem"
#tworzenie tabeli flags
#sprawdzenie czy wartosci sa poza sensownym zakresem np. wiek nie moze byc ujemny lub >120 lat
def range_flags(frame: pd.DataFrame) -> pd.DataFrame:
    flags = pd.DataFrame(index=frame.index)

    if "age" in frame.columns:
        flags["age_out"] = (frame["age"] < 0) | (frame["age"] > 120)

    if "bmi" in frame.columns:
        flags["bmi_out"] = (frame["bmi"] < 10) | (frame["bmi"] > 80)

    if "HbA1c_level" in frame.columns:
        flags["hba1c_out"] = (frame["HbA1c_level"] < 3) | (frame["HbA1c_level"] > 20)

    if "blood_glucose_level" in frame.columns:
        flags["glucose_out"] = (frame["blood_glucose_level"] < 40) | (frame["blood_glucose_level"] > 600)

    return flags

flags = range_flags(df)
flag_counts = flags.sum().sort_values(ascending=False)

print("Ile wartości jest poza zakresem?")
display(flag_counts[flag_counts > 0])

In [ ]:
 #Strategia: zamiast usuwać dane, wykonujemy "clipping" do sensownego zakresu.
# To jest często lepsze w praktyce (mniejsza utrata danych).
def clip_ranges(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    if "age" in out.columns:
        out["age"] = out["age"].clip(0, 120)
    if "bmi" in out.columns:
        out["bmi"] = out["bmi"].clip(10, 80)
    if "HbA1c_level" in out.columns:
        out["HbA1c_level"] = out["HbA1c_level"].clip(3, 20)
    if "blood_glucose_level" in out.columns:
        out["blood_glucose_level"] = out["blood_glucose_level"].clip(40, 600)
    return out

df = clip_ranges(df)

Analiza targetu diabates:


*   Ile przypadkow jest 0 vs 1
*   czy dataset jest zbalansowany
*   accuracy
*   jakie metryki są lepsze




In [ ]:
target_col = "diabetes"
if target_col not in df.columns:
    raise ValueError("Nie znaleziono kolumny 'diabetes' w danych.")
#value_counts() liczy ile jest 0 i 1, normalize=true - %
counts = df[target_col].value_counts()
pct = df[target_col].value_counts(normalize=True) * 100

print(" Rozkład targetu (diabetes)")
display(pd.DataFrame({"count": counts, "pct": pct.round(2)}))

plt.figure(figsize=(5, 3))
#słupki ile 0 i 1
sns.countplot(x=target_col, data=df)
plt.title("Rozkład targetu (0 = brak cukrzycy, 1 = cukrzyca)")
plt.show()

Feature typing - podział cech na numeryczne i kategoryczne

In [ ]:
#lista kolumn oprocz targetu
feature_cols = [c for c in df.columns if c != target_col]
#wybiera z nich tylko numeryczne
num_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
#reszta kategorie tekstowe
cat_cols = [c for c in feature_cols if c not in num_cols]

print("Cechy numeryczne:", num_cols)
print("Cechy kategoryczne:", cat_cols)

Rozkład cech numerycznych


*   Histogramy cech numerycznych
*   KDE(gęstość) rozdzielone na target
*   countploty dla kategorii



In [ ]:
# Histogramy
if len(num_cols) > 0:
  #rysowanie histogram dla kazdej kolumny numerycznej
    df[num_cols].hist(figsize=(12, 8), bins=30)
    plt.suptitle("Rozkłady cech numerycznych", y=1.02)
    plt.tight_layout()
    plt.show()

# porównanie rozkładów w diabetes=0 vs 1
for c in num_cols:
    plt.figure(figsize=(6, 3))
    #rysowanie wygładzonego rozkładu, osobno dla diabetes = 0 i diabetes= 1
    sns.kdeplot(data=df, x=c, hue=target_col, common_norm=False)
    plt.title(f"KDE: {c} vs diabetes")
    plt.show()

In [ ]:
for c in cat_cols:
    plt.figure(figsize=(7, 3))
    #oder - ustawienie kategorii od najczęstszej, rysuje słupki ile jest każdej kategorii
    order = df[c].value_counts().index
    sns.countplot(data=df, x=c, order=order)
    plt.title(f"Liczności kategorii: {c}")
    plt.xticks(rotation=30, ha="right")
    plt.show()

Ryzyko warunkowe


*   Jaki odsetek cukrzycy w każdej kategorii


In [ ]:
for c in cat_cols:
    # średnia z diabetes (0/1) = odsetek 1
    #target_col to 0/1, srednia z 0/1 = odsetek 1 czyli % cukrzycy to jest  doklasnie p(diabetes=1|dana kategoria)
    rate = df.groupby(c)[target_col].mean().sort_values(ascending=False) * 100
    rate_df = rate.rename("diabetes_rate_%").to_frame()

    print(f"\n Odsetek cukrzycy (%) wg {c} ")
    display(rate_df)

    plt.figure(figsize=(7, 3))
    sns.barplot(x=rate_df.index, y=rate_df["diabetes_rate_%"])
    plt.title(f"Odsetek cukrzycy (%) wedlug {c}")
    plt.xticks(rotation=30, ha="right")
    plt.ylim(0, max(5, rate_df["diabetes_rate_%"].max() * 1.15))
    plt.show()

Outliery i różnice między grupami (boxplot)
W medycynie outliery mogą oznaczać: błąd pomiaru, rzadki przypadek kliniczny, bład wpisu


*   Sprawdzenie outlierów metodą IQR
*   Boxplot dla diabetes = 0/1 - czy grupy różnią się widocznie



In [ ]:
#funkcja liczy kwartale q1 i q3, IQR = Q3 - Q1, outlier jesli < q1-1.5/qr lub >q3+1.5iqr zwraca % outlierów
def iqr_outlier_pct(s: pd.Series) -> float:
    s = s.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return float(((s < lo) | (s > hi)).mean() * 100)

if len(num_cols) > 0:
    outlier_pct = pd.Series({c: iqr_outlier_pct(df[c]) for c in num_cols}).sort_values(ascending=False)
    print(" Odsetek outlierów (IQR) w % ")
    display(outlier_pct.to_frame("outlier_pct_%"))

for c in num_cols:
    plt.figure(figsize=(5, 3))
    #pokazanie rozkladu cech w 2 roznych grupach 0 vs 1
    #latwo zobaczyc mediany, rozrzut, outliery
    sns.boxplot(data=df, x=target_col, y=c)
    plt.title(f"{c} vs diabetes (boxplot)")
    plt.show()

Korelacja i interpretacja

In [ ]:
if len(num_cols) > 1:
  #corr liczy korelacje miedzy cechami
    corr = df[num_cols + [target_col]].corr(numeric_only=True)

    plt.figure(figsize=(10, 7))
    sns.heatmap(corr, annot=True, fmt=".2f")
    plt.title("Heatmap korelacji (cechy numeryczne + target)")
    plt.show()

Interakcja Age x BMI


*   Jak ryzyo cukrzycy zmienia się jednocześnie z wiekiem i BMI



In [ ]:
if ("age" in df.columns) and ("bmi" in df.columns):
    tmp = df.copy()
#dzieli wiek i BMI na przedzialy (binning)
    tmp["age_bin"] = pd.cut(tmp["age"], bins=[0, 30, 45, 60, 75, 120], include_lowest=True)
    tmp["bmi_bin"] = pd.cut(tmp["bmi"], bins=[10, 18.5, 25, 30, 35, 80], include_lowest=True)
#robi tabele : wiersze - przedzialy wieku, kolumny = przedzialy BMI, wartosci = % cukrzycy
    pivot = tmp.pivot_table(index="age_bin", columns="bmi_bin", values=target_col, aggfunc="mean") * 100

    plt.figure(figsize=(10, 4))
    sns.heatmap(pivot, annot=False)
    plt.title("Odsetek cukrzycy (%) — Age × BMI")
    plt.xlabel("BMI (przedziały)")
    plt.ylabel("Wiek (przedziały)")
    plt.show()

Testy statystyczne: Welch t-test i Cohen's d dla cech numerycznych


In [ ]:
from scipy import stats

def cohens_d(a, b) -> float:
    a = np.asarray(a); b = np.asarray(b)
    a = a[~np.isnan(a)]; b = b[~np.isnan(b)]
    na, nb = len(a), len(b)
    sa, sb = a.std(ddof=1), b.std(ddof=1)
    sp = np.sqrt(((na - 1) * sa**2 + (nb - 1) * sb**2) / (na + nb - 2))
    return float((a.mean() - b.mean()) / sp) if sp > 0 else np.nan

stat_rows = []
for c in num_cols:
    g0 = df.loc[df[target_col] == 0, c].dropna()
    g1 = df.loc[df[target_col] == 1, c].dropna()

    # Minimalna liczność, żeby test miał sens
    if len(g0) > 20 and len(g1) > 20:
      #porownanie srednie w diabetes= 0 vs diabetes=1
      #equal_var = false - Welch bez zalozenia głównych wariancji
        t_stat, p_val = stats.ttest_ind(g0, g1, equal_var=False)
      #Cohen's d pokazuje jak duza jest roznica
        d = cohens_d(g0, g1)
        stat_rows.append((c, g0.mean(), g1.mean(), t_stat, p_val, d))
#stats_df - zbiera wyniki dla kazdej cechy
stats_df = pd.DataFrame(
    stat_rows,
    columns=["feature", "mean_no_diabetes", "mean_diabetes", "t_stat", "p_value", "cohens_d"]
).sort_values("p_value")

print("Testy statystyczne: Welch t-test + Cohen’s d")
display(stats_df)

print("\nInterpretacja Cohen’s d:")
print("- ~0.2 mały efekt")
print("- ~0.5 średni efekt")
print("- ~0.8 duży efekt")

Feature engineering
Dodanie cech : BMI(WHO-like), flagi wysokiego ryzyka: glukoza i HbA1c, risk score (heurystyka)

In [ ]:
#robienie kopii by nie popsuc oruginalnego df
df_fe = df.copy()

# BMI kategorie
if "bmi" in df_fe.columns:
  #zmienia bmi na kategorie(underweight,normal, ...)
    df_fe["bmi_cat"] = pd.cut(
        df_fe["bmi"],
        bins=[0, 18.5, 25, 30, 35, 80],
        labels=["underweight", "normal", "overweight", "obese_I", "obese_II+"]
    )

# Flagi kliniczne (progi poglądowe)
if "blood_glucose_level" in df_fe.columns:
    df_fe["high_glucose_flag"] = (df_fe["blood_glucose_level"] >= 200).astype(int)

if "HbA1c_level" in df_fe.columns:
    df_fe["high_hba1c_flag"] = (df_fe["HbA1c_level"] >= 6.5).astype(int)

# Prosty risk_score (heurystyka do EDA; nie jest modelem klinicznym)
needed = {"age", "bmi", "HbA1c_level", "blood_glucose_level"}
if needed.issubset(df_fe.columns):
  #risk score - suma wazona
    df_fe["risk_score"] = (
        df_fe["age"] * 0.02
        + df_fe["bmi"] * 0.03
        + df_fe["HbA1c_level"] * 0.55
        + (df_fe["blood_glucose_level"] / 100) * 0.35
    )

print(" Podgląd po feature engineering")
display(df_fe.head())

# Sprawdzenie ryzyka po kategoriach
if "bmi_cat" in df_fe.columns:
    print("\nOdsetek cukrzycy (%) wedlug bmi_cat:")
    display((df_fe.groupby("bmi_cat")[target_col].mean() * 100).rename("diabetes_rate_%").to_frame())

if "high_hba1c_flag" in df_fe.columns:
    print("\nOdsetek cukrzycy (%) wg high_hba1c_flag:")
    display((df_fe.groupby("high_hba1c_flag")[target_col].mean() * 100).rename("diabetes_rate_%").to_frame())

ML + EWALUACJA

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)
#x-cechy, y -target
X = df_fe.drop(columns=[target_col])
y = df_fe[target_col].astype(int)

# One-hot encoding kategorii - zamiana kategorii na liczby
X = pd.get_dummies(X, drop_first=True)

# Podział z zachowaniem proporcji klas
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

clf = LogisticRegression(max_iter=3000, class_weight="balanced")
clf.fit(X_train, y_train)
#predict_proba - daje prawdopodobienstwo klasy 1, próg 0.5 robi z tego 0/1
proba = clf.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print(" Classification report (threshold=0.5)")
print(classification_report(y_test, pred, digits=4))

print("ROC-AUC:", roc_auc_score(y_test, proba))
print("PR-AUC (Average Precision):", average_precision_score(y_test, proba))

cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt="d")
plt.title("Confusion Matrix (threshold=0.5)")
plt.xlabel("Predykcja")
plt.ylabel("Rzeczywiste")
plt.show()

Dobór progu(threshold tuning)

In [ ]:
from sklearn.metrics import precision_recall_curve
#liczenie precision, recall dla różnych progów
prec, rec, thr = precision_recall_curve(y_test, proba)

plt.figure(figsize=(5, 4))
plt.plot(rec, prec)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall curve")
plt.show()
#szukanie progów gdzie recall >=0.8
target_recall = 0.80
idx = np.where(rec >= target_recall)[0]

if len(idx) > 0:
    chosen_idx = idx[-1]
    chosen_thr = thr[max(chosen_idx - 1, 0)] if chosen_idx > 0 else 0.5

    print(f"Wybrany próg dla recall >= {target_recall:.2f}: {chosen_thr:.4f}")

    pred_thr = (proba >= chosen_thr).astype(int)
    print("\n Classification report (tuned threshold)")
    print(classification_report(y_test, pred_thr, digits=4))
else:
    print("Nie udało się osiągnąć docelowego recall dla dostępnych progów.")

Permutation Importance

In [ ]:
from sklearn.inspection import permutation_importance
#miesza 1 kolumnę i patrzy jak spada ROC-AUC
#większy spadek = cecha bardziej wazna
perm = permutation_importance(
    clf, X_test, y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    scoring="roc_auc"
)

imp = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)

print("Top 20 cech (Permutation Importance, ROC-AUC)")
display(imp.head(20).to_frame("perm_importance_mean"))

plt.figure(figsize=(7, 6))
imp.head(20).sort_values().plot(kind="barh")
plt.title("Top 20 cech — Permutation Importance")
plt.show()

In [ ]:
from sklearn.calibration import calibration_curve
#dzielenie przewidywanych prawdopodobienstw na 10 koszykow
#sprawdzenie czy przewidziane 0.7 faktycznie jest to ~70% pozytywnych
prob_true, prob_pred = calibration_curve(y_test, proba, n_bins=10)

plt.figure(figsize=(5, 4))
plt.plot(prob_pred, prob_true, marker="o")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("Średnie przewidywane prawdopodobieństwo")
plt.ylabel("Rzeczywisty odsetek pozytywnych")
plt.title("Calibration curve (LogReg baseline)")
plt.show()

In [ ]:
out_file = "diabetes_clean_engineered.csv"
#zapis do pliku, index=false by nie dodac kolumny z indeksem
df_fe.to_csv(out_file, index=False)
print(f" Zapisano plik: {out_file}")